# Chương 2: Các Kỹ Thuật Học Máy Cơ Bản & Mạng Nơ-ron Đa Tầng (MLP)
**Học viện Công nghệ Bưu chính Viễn thông (PTIT)**
*Môn học: Phát triển các Hệ thống Thông minh (GVHD: PGS.TS Trần Đình Quế)*
*Sinh viên: Phạm Văn Tư - B23DCCN879*

## 1. Khảo Sát & Tiền Xử Lý Dữ Liệu (EDA & Preprocessing)
Chúng ta khảo sát 3 bài toán thực nghiệm lớn: Sàng lọc Đái tháo đường (Phân loại nhị phân), Định giá Bất động sản (Hồi quy) và Phân loại Đánh giá Khách hàng (NLP Sentiment).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

print("✓ Đã nạp thành công các thư viện Scikit-Learn, NumPy, Pandas!")


✓ Đã nạp thành công các thư viện Scikit-Learn, NumPy, Pandas!


## 2. Huấn Luyện Các Thuật Toán Học Máy Cơ Bản
Tiến hành huấn luyện và in log chu kỳ định kỳ để kiểm soát quá trình tối ưu:

In [2]:
# Mô phỏng tập dữ liệu sàng lọc đái tháo đường
np.random.seed(42)
X_diabetes = np.random.randn(5000, 10)
y_diabetes = (X_diabetes[:, 0]*1.5 + X_diabetes[:, 1]*0.8 - X_diabetes[:, 2]*0.5 + np.random.randn(5000)*0.5 > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_diabetes, y_diabetes, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=100)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)
print("Độ chính xác Logistic Regression:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Bình thường', 'Nguy cơ']))


Độ chính xác Logistic Regression: 0.901
              precision    recall  f1-score   support

 Bình thường       0.92      0.88      0.90       514
     Nguy cơ       0.88      0.92      0.90       486

    accuracy                           0.90      1000
   macro avg       0.90      0.90      0.90      1000
weighted avg       0.90      0.90      0.90      1000



## 3. Mạng Nơ-ron Đa Tầng (MLP) Scratch Bằng NumPy Thuần
Hiện thực hóa forward pass, backpropagation tính đạo hàm chuỗi ma trận:

In [3]:
class MLPScratch:
    def __init__(self, in_dim, hidden_dim, out_dim, lr=0.01):
        self.W1 = np.random.randn(in_dim, hidden_dim) * np.sqrt(2.0 / in_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, out_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros((1, out_dim))
        self.lr = lr

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = np.maximum(0, self.Z1) # ReLU
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.probs = 1.0 / (1.0 + np.exp(-np.clip(self.Z2, -15, 15))) # Sigmoid
        return self.probs

    def backward(self, X, y):
        m = X.shape[0]
        dZ2 = self.probs - y.reshape(-1, 1)
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * (self.Z1 > 0)
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

mlp = MLPScratch(in_dim=10, hidden_dim=32, out_dim=1, lr=0.05)
for epoch in range(1, 101):
    probs = mlp.forward(X_train_scaled)
    mlp.backward(X_train_scaled, y_train)
    if epoch % 20 == 0:
        loss = -np.mean(y_train * np.log(probs + 1e-8) + (1 - y_train) * np.log(1 - probs + 1e-8))
        print(f"Epoch {epoch:3d}/100 - Loss: {loss:.4f}")


Epoch  20/100 - Loss: 0.8489


Epoch  40/100 - Loss: 0.9108
Epoch  60/100 - Loss: 0.9833


Epoch  80/100 - Loss: 1.0562


Epoch 100/100 - Loss: 1.1269
